# Predicción de Fuga de Talento — EDA y Entrenamiento
**Dataset:** IBM HR Analytics Employee Attrition  
**Descarga:** https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset  

Pasos:
1. Carga y exploración del dataset
2. Análisis exploratorio (EDA)
3. Preprocesamiento
4. Entrenamiento del modelo
5. Evaluación
6. Exportar `model.pkl`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', palette='muted')
print('Librerías cargadas correctamente ✓')

## 1. Carga del Dataset

**Opción A — Kaggle (requiere cuenta):**  
Ir a https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset  
→ Download → descomprimir → copiar `WA_Fn-UseC_-HR-Employee-Attrition.csv` en esta carpeta `notebooks/data/`

**Opción B — Kaggle CLI:**  
```bash
pip install kaggle
kaggle datasets download -d pavansubhasht/ibm-hr-analytics-attrition-dataset
unzip ibm-hr-analytics-attrition-dataset.zip -d data/
```

**Opción C — GitHub (sin cuenta):**  
El CSV también está disponible en varios repos públicos:  
https://raw.githubusercontent.com/dsrscientist/dataset1/master/HR_comma_sep.csv  
(este es el dataset de Kaggle HR Analytics — distinto al IBM pero similar estructura)

In [ ]:
df = pd.read_csv('data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Tipos de datos:')
print(df.dtypes)
print(f'\nValores nulos:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

## 2. Análisis Exploratorio (EDA)

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

attrition_counts = df['Attrition'].value_counts()
axes[0].pie(attrition_counts, labels=attrition_counts.index, autopct='%1.1f%%',
            colors=['#3b82f6', '#ef4444'])
axes[0].set_title('Distribución de Attrition')

sns.countplot(data=df, x='Attrition', ax=axes[1], palette=['#3b82f6', '#ef4444'])
axes[1].set_title('Conteo por clase')
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom')
plt.tight_layout()
plt.show()
print(f'\nDesbalance de clases: {attrition_counts.to_dict()}')

In [ ]:
# Variables numéricas vs Attrition
numeric_cols = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'DistanceFromHome',
                'JobSatisfaction', 'WorkLifeBalance', 'NumCompaniesWorked']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    df.boxplot(column=col, by='Attrition', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('Attrition')

axes[-1].set_visible(False)
plt.suptitle('Variables Numéricas vs Attrition')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de correlaciones
df_encoded = df.copy()
df_encoded['Attrition_num'] = (df['Attrition'] == 'Yes').astype(int)
df_encoded['OverTime_num'] = (df['OverTime'] == 'Yes').astype(int)

corr_cols = ['Attrition_num', 'Age', 'MonthlyIncome', 'YearsAtCompany',
             'JobSatisfaction', 'EnvironmentSatisfaction', 'WorkLifeBalance',
             'NumCompaniesWorked', 'OverTime_num', 'DistanceFromHome',
             'YearsSinceLastPromotion', 'PerformanceRating']

plt.figure(figsize=(12, 8))
sns.heatmap(df_encoded[corr_cols].corr(), annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, square=True)
plt.title('Correlación de variables con Attrition')
plt.tight_layout()
plt.show()

## 3. Preprocesamiento

In [ ]:
# Selección de features (las mismas que expone el endpoint /predict)
FEATURES = [
    'Age', 'MonthlyIncome', 'YearsAtCompany', 'YearsInCurrentRole',
    'YearsSinceLastPromotion', 'JobSatisfaction', 'EnvironmentSatisfaction',
    'WorkLifeBalance', 'NumCompaniesWorked', 'OverTime',
    'DistanceFromHome', 'PerformanceRating'
]
TARGET = 'Attrition'

df_model = df[FEATURES + [TARGET]].copy()

# Encodings binarios
df_model['OverTime'] = (df_model['OverTime'] == 'Yes').astype(int)
df_model[TARGET] = (df_model[TARGET] == 'Yes').astype(int)

X = df_model[FEATURES]
y = df_model[TARGET]

print(f'Features: {FEATURES}')
print(f'Shape X: {X.shape}, Shape y: {y.shape}')
print(f'Attrition = 1: {y.sum()} ({y.mean()*100:.1f}%)')

In [ ]:
# Split estratificado para mantener proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras')

## 4. Entrenamiento — Random Forest

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',   # compensa el desbalance de clases
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Cross-validation
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
print(f'AUC-ROC (CV 5-fold): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

## 5. Evaluación

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Se queda', 'Se va']))
print(f'AUC-ROC (test): {roc_auc_score(y_test, y_proba):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Se queda', 'Se va'],
    colorbar=False, ax=axes[0]
)
axes[0].set_title('Matriz de Confusión')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#3b82f6', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set(xlabel='FPR', ylabel='TPR', title='Curva ROC')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance — clave para explicar el modelo en la tesis
importances = pd.Series(model.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='#3b82f6')
plt.title('Importancia de Variables — Random Forest')
plt.xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

print('\nTop 5 variables más importantes:')
print(importances.sort_values(ascending=False).head())

## 6. Exportar el modelo
Guardamos el modelo en `model/model.pkl`. El microservicio lo cargará automáticamente.

In [ ]:
import os
os.makedirs('../model', exist_ok=True)
joblib.dump(model, '../model/model.pkl')
print('Modelo guardado en ../model/model.pkl ✓')
print(f'Tamaño: {os.path.getsize("../model/model.pkl") / 1024:.1f} KB')